### 6. Qual o percentual de processos de Recuperação Extrajudicial que obtêm homologação do plano sem impugnação expressiva de credores?


**Pergunta:** a recuperação extrajudicial (RE, art. 161-167 da Lei
11.101/2005) é o procedimento "rápido e confidencial": os credores assinam
um plano diretamente com a empresa e o juiz apenas homologa. A pergunta
mede quantos desses processos chegam à homologação — e sem disputa.

**Fonte:** 
`gold_base_analitica` — classe 128 (RE originária):
   `data_homologacao_plano` (marco da homologação, art. 162),
   `data_decretao_falencia`,
   `data_encerramento`, 
   `cenario_desfecho` (desfechos),
   `qtd_impugnacoes` (impugnações de crédito nos autos) e
   `data_ajuizamento`.

Este notebook não refaz cálculos: consome o resultado pronto.

**Recorte:** classe 128, ajuizado a partir de 09/06/2005 (o instituto não
existia na lei anterior).

**Correção do universo (nota metodológica):** O recorte
pela classe processual (128) isola as REs originárias (~300
processos — compatível com a baixa demanda real do instituto). 

**Sobre a "impugnação expressiva":** o Datajud não
registra se os credores contestam o plano, nem o tamanho da contestação — nem sequer há movimento nomeado para a homologação do plano (mesma limitação estrutural da pergunta 2). O único substituto possível é a ausência de impugnações de crédito nos autos — e a leitura correta dele é "não há contestação registrada", não "os credores aprovaram".

**Limitações:**
1. A homologação do plano não tem movimento próprio na TPU — se não for registrada, a pergunta fica sem denominador;
2. O instituto é de baixa demanda (a confidencialidade é seu atrativo) — com amostras pequenas, as taxas exigem cautela;
3. Pedidos de RE arquivados podem ter migrado para RJ judicial (sem vínculo pai-filho no Datajud) — não rastreável;
4. **O contador de impugnações não distingue dois atos diferentes.** A
   lei prevê: (i) a **objeção ao plano** (arts. 162-163) — o credor
   discorda do *conteúdo do plano* antes da homologação, e a objeção
   suspende o processo de homologação; e (ii) a **impugnação de
   crédito** — o credor diverge da *lista de créditos*, etapa de
   verificação que corre à parte e não afeta o plano. O contador
   `qtd_impugnacoes` mede apenas o **segundo**. Consequência: mesmo
   que as homologações fossem detectáveis, uma RE "com impugnação
   registrada" poderia ter tido o plano **aceito sem qualquer objeção**
   — a impugnação pode se referir só ao valor de um crédito na lista.
   O indicador "homologado sem impugnação expressiva" ficaria
   contaminado por falsos positivos — por isso o proxy é frágil
   mesmo no cenário em que a homologação fosse registrável.

**O que a consulta a seguir mede:** o quadro de desfechos das REs originárias
(classe 128) e, nele, o marco que define a resposta — a homologação do
plano (art. 162 da Lei 11.101/2005). Uma linha com o universo total, quantos têm
homologação registrada, quantos convoluiram em falência, quantos
encerraram sem convolação, quantos seguem em andamento, os percentuais
de cada bloco e a mediana de tramitação dos processos de em andamento (até a data
download).

**Entrada:** apenas colunas da gold — `data_homologacao_plano`,
`data_decretao_falencia`, `data_encerramento`, `data_ajuizamento`.

**Como funciona por dentro:** os três desfechos são blocos mutuamente
exclusivos definidos por `CASE`: convolação (falência registrada),
encerramento sem convolação (encerramento registrado, falência não) e
em andamento (nenhum marco de fim). Os percentuais são cada bloco
sobre o total; a mediana de dias usa `datediff` até a data de corte
(01/09/2026) **somente para os em andamento** — os concluídos têm
medida própria na célula seguinte.

**Saída esperada e papel na resposta:** se `com_homologacao` vier
zero, o denominador da taxa pedida pela pergunta é nulo — e não será possivel responder a pergunta por esse caminho.

In [0]:
%sql
-- Q6 · Quadro de desfechos das REs (classe 128) + marco de homologação
SELECT
  count(*)                                                                AS total_re,
  count(CASE WHEN data_homologacao_plano IS NOT NULL THEN 1 END)          AS com_homologacao,
  count(CASE WHEN data_decretao_falencia IS NOT NULL THEN 1 END)          AS convoluidas_falencia,
  count(CASE WHEN data_decretao_falencia IS NULL
              AND data_encerramento IS NOT NULL THEN 1 END)               AS encerradas_sem_convolacao,
  count(CASE WHEN data_decretao_falencia IS NULL
              AND data_encerramento IS NULL THEN 1 END)                   AS em_andamento,
  round(count(CASE WHEN data_decretao_falencia IS NOT NULL THEN 1 END)
        * 100.0 / count(*), 1)                                            AS pct_convoluidas,
  round(count(CASE WHEN data_decretao_falencia IS NULL
              AND data_encerramento IS NOT NULL THEN 1 END)
        * 100.0 / count(*), 1)                                            AS pct_encerradas,
  round(count(CASE WHEN data_decretao_falencia IS NULL
              AND data_encerramento IS NULL THEN 1 END)
        * 100.0 / count(*), 1)                                            AS pct_em_andamento,
  round(percentile_approx(CASE WHEN data_decretao_falencia IS NULL
                              AND data_encerramento IS NULL
                              THEN datediff(date('2026-09-01'), data_ajuizamento)
                         END, 0.5), 1)                                    AS mediana_dias_em_andamento_ate_corte
FROM default.gold_base_analitica
WHERE classe_codigo = 128 AND escopo_lei_atual;

**O que a consulta a seguir mede:** o detalhamento do mesmo universo de RE por
desfecho consolidado (`cenario_desfecho`), cruzando três dimensões —
quantos processos existem em cada desfecho (ativo, ), quão rápido terminaram (mediana de
duração) e quantas impugnações de crédito carregam (média e contagem
de autos com ao menos uma).

**Entrada:** colunas da gold — `cenario_desfecho`,
`dias_duracao_processo_total`, `qtd_impugnacoes`.

**Como funciona por dentro:** agregação `GROUP BY` no desfecho com
`percentile_approx(..., 0.5)` para a mediana (robusta a extremos,
padrão do projeto) e `avg` para a média de impugnações;
`com_impugnacao` conta os autos com impugnação, que é a leitura
correta do contador (ocorrência, não volume distribuído).

**Saída esperada e papel na resposta:** 

(i) A duração
mediana dos encerrados testa se o fim registrado é compatível com o
procedimento homologatório cumprido (~1 ano) — a evidência de que as
homologações existiram mas não foram registradas; 

  (ii) a coluna de
impugnações testa o único substituto disponível para a "impugnação
expressiva" — e o teste falha por duas razões independentes:

- O contador registra disputa de *lista de
  créditos*, não objeção ao *plano* (ver limitação 4) — um valor alto  não provaria contestação do plano;

- Espera-se pouquíssimos autos com
  impugnação (e o output confirma). Uma variável zerada em ~96% do
  universo não distingue grupo algum: todo processo sem impugnação
  registrada apareceria como "credores aprovaram", quando na verdade
  significa apenas "nada foi registrado".

Mesmo que as homologações fossem detectáveis, este proxy não
sustentaria o indicador pedido — a ausência de registro não é prova
de aprovação.

In [0]:
%sql
-- Q6 · Detalhamento: desfecho × duração × impugnações
SELECT cenario_desfecho,
       count(*)                                                     AS n,
       round(percentile_approx(dias_duracao_processo_total, 0.5), 1) AS mediana_duracao_dias,
       round(avg(qtd_impugnacoes), 1)                                AS media_impugnacoes,
       count(CASE WHEN qtd_impugnacoes > 0 THEN 1 END)               AS com_impugnacao
FROM default.gold_base_analitica
WHERE classe_codigo = 128 AND escopo_lei_atual
GROUP BY 1
ORDER BY n DESC;

**O que a consulta a seguir faz:** verificação de coerência (QA) — não responde
à pergunta, mas valida a semântica do mart contra ela. O stay period
(art. 6º, §4º) termina em homologação do plano **ou** em falência; se
a classe 128 registra zero homologações e zero falências (saída da
célula 1), então `data_fim_stay` deve ser nulo em **todas** as linhas.

**Entrada:** uma coluna da gold — `data_fim_stay`.

**Como funciona por dentro:** contagem simples de linhas com e sem o
marco, sobre o mesmo recorte da célula 1.

**Leitura do output:** o valor esperado é `com_fim_stay = 0`. Se vier
diferente de zero, há inconsistência entre a cascata de marcos do
Bloco 4 do notebook 03_golden_modelagem  e o quadro de desfechos — o que exigiria investigação antes de
qualquer relato. Zero confirma que a regra de construção do stay e os
desfechos contados contam a mesma história, de ponta a ponta.

In [0]:
%sql
-- Q6 · Coerência com a semântica do stay (art. 6º, §4º)
-- O stay termina em homologação OU falência. Se classe 128 tem 0 falências
-- e 0 homologações detectadas, data_fim_stay deve ser nulo em TODAS as linhas.
SELECT count(*)                                        AS total_re,
       count(CASE WHEN data_fim_stay IS NOT NULL THEN 1 END) AS com_fim_stay  -- esperado: 0
FROM default.gold_base_analitica
WHERE classe_codigo = 128 AND escopo_lei_atual;

#### Análises das respostas

**Quadro de desfechos das REs originárias (classe 128, Lei 11.101/2005):**
| Indicador | Valor |
|---|---|
| Processos de RE originária | 305 |
| Homologações de plano identificadas | **0** |
| Convoluções em falência | 0 (0,0%) |
| Encerradas sem convolação (encerramento/extinção registrados) | 60 (19,7%) |
| Em andamento / sem marco de fim | 245 (80,3%) |
| Mediana de tramitação das em andamento (até 01/09/2026) | 804 dias (~2,2 anos) |
| Taxa de homologação | **0,00%** |
| % de homologados sem impugnação | não calculável (denominador nulo) |

**Decomposição por desfecho consolidado (mart):** 241 ativos, 60 extintos
(mediana de 377 dias de tramitação) e 4 encerrados (mediana de 477 dias).
Os 60 do quadro acima correspondem aos autos que já têm marco de fim
registrado; a diferença entre os cortes é a regra de consolidação do
desfecho (qual evento veio primeiro), não divergência de dados.

**A pergunta é NÃO-RESPONDÍVEL no padrão Datajud.** Nem um único marco de homologação de plano extrajudicial foi
identificado em 305 processos da classe própria. O achado espelha a
limitação estrutural da pergunta 2 (nenhum movimento com a palavra "plano" em ~14,1
milhões de eventos): a homologação do art. 162 — ato final de todo o
procedimento — não possui movimento nomeado nem código TPU, e nenhum
tribunal do corpus a registra de forma detectável. Sem o marco, não
existe denominador para a taxa de homologação sem impugnação.

**Evidência de homologações ocultas:** 60 REs já chegaram ao fim, com
tramitação típica de ~1 ano (377–477 dias) — duração compatível com
procedimento homologatório cumprido, não com pedidos abandonados. Como a
execução do plano homologado é o desfecho natural do instituto
(arts. 162-167), é altamente improvável que dezenas de procedimentos
tenham sido encerrados **sem** o plano ter sido homologado em algum
momento. O resultado é, portanto, **positivamente mascarado**: a fonte
registra o fim do ciclo (encerramento/extinção) mas não o ato central
(homologação). O gradiente de registrabilidade fica completo: pedido
registrado (305), fim do ciclo registrado (60), homologação invisível (0).

**Impugnações sem sinal utilizável:** apenas 9 dos 305 autos têm alguma
impugnação de crédito (média ~0 em todos os desfechos) — mesmo como proxy,
a variável não separa impugnação ao plano (art. 163, §1º) da verificação
ordinária de créditos.

**convolação zero é coerente:** a RE é procedimento homologatório e
consensual — não prevê decretação de falência como desfecho (ao contrário
da RJ, art. 73). A ausência total de convolação confirma que a classe 128
está limpa de contaminação por autos conexos.

**Coerência com o mart:** `data_fim_stay` nulo em todas as 305 linhas
(o stay só termina em homologação ou falência — nenhuma das duas é
registrada na classe 128), confirmando a semântica do Bloco 4 de ponta a
ponta.

**Leitura jurismétrica:** a RE, desenhada pela Lei 11.101/2005 justamente
pela celeridade e confidencialidade, é praticamente invisível na base
pública nacional — o instituto é de baixa demanda (80,3% ainda sem marco
de fim) e seu ato central não é registrável. Para estudá-lo, a fonte
adequada são os portões de transparência dos próprios tribunais, não o
Datajud.

#### Resposta final (pergunta 6)

O percentual de recuperações extrajudiciais com homologação do plano
**não pode ser calculado no padrão Datajud: entre os 305 processos de RE
originária, zero homologações são registráveis** — embora 60 processos
estejam encerrados, com tramitação típica de cerca de 1 ano compatível
com o procedimento cumprido, o ato que define o instituto (a homologação,
art. 162) não tem movimento próprio nem código na TPU, e nenhum tribunal
o registra de forma detectável. A resposta estrutural é tripla: (i) o
percentual pedido é não-respondível — e a ausência é o achado; (ii) a
invisibilidade é positiva, não casual: processos terminam sem que a
homologação apareça — o dado mascara atos que sabidamente ocorreram;
(iii) o instituto consensual, criado para ser rápido e confidencial, é
praticamente invisível na base pública — para medi-lo, a fonte adequada
são os portões de transparência dos tribunais, não o Datajud. A taxa de
"homologação sem impugnação" permanece, portanto, fora dos limites de
rastreabilidade da base.